# 📝 과제 02 정답: 파이썬 퀴즈 자동 채점을 DeepEval 테스트로 굳히기

## 만들 결과

부트캠프 운영팀은 매주 보는 파이썬 퀴즈(단답·서술)의 1차 채점을 판사 모델에게 맡기려 합니다. 퀴즈가 바뀌어도 같은 채점을 반복할 수 있게 채점 기준을 DeepEval 테스트 케이스와 지표로 굳히고, 조교가 채점한 답안 36개(단답 18·서술 18)의 교사 판정과 대조해 문항 유형마다 채점 방법을 고릅니다.

- 답안 36개를 `LLMTestCase`로 담습니다.
- 단답은 **코드 규칙**(문자열 비교)과 **단답 G-Eval**을 비교합니다.
- 서술은 rubric G-Eval을 **기준 답변 없이·있이** 비교합니다.
- 방법별 판사 성적표로 문항 유형마다 채점 방법을 고르고, 평가 범위표로 다음 주에 보탤 답안 유형을 고릅니다.
- 결정 기록을 남깁니다.

교안 02의 G-Eval·rubric(3절), `evaluate`와 판사 성적표(6절), 평가 범위표(7절)를 퀴즈 채점에 적용합니다. 과제 01과 같은 답안을 쓰지만 과제 01 결과는 읽지 않습니다.

| 방법 | 문항 유형 | 채점에 쓰는 것 | 판사 호출 |
|---|---|---|---|
| 코드 규칙 | 단답 | 공백을 지운 답안이 기준 답변과 같은지 | 0번 |
| 단답 G-Eval | 단답 | 문항·답안·기준 답변 + 단답 채점 규칙 | 18번 |
| 서술 G-Eval(기준 답변 없음) | 서술 | 문항·답안·채점 항목 A·B + rubric | 18번 |
| 서술 G-Eval | 서술 | 위와 같음 + 기준 답변 | 18번 |

## 준비: 데이터

- **자료:** `data/과제/python_quiz_set.json`. 문항 6개(단답 K1–K3, 서술 E1–E3), 문항마다 답안 6개와 교사 판정(pass 3·fail 3), 교사 채점 규칙 `grading_rules`, 평가 범위 목표 `coverage_targets`가 있습니다. 수업용 가상 답안입니다.
- **순서:** 테스트 케이스 → 코드 규칙 → 단답 G-Eval → 서술 G-Eval 두 개 → `evaluate` → 성적표 → 평가 범위표 → 방법 고르기 → 결정 기록
- **작성:** `[작성]` 셀의 `...`를 번호 주석대로 채우고 바로 아래 `[자가채점]` 셀을 실행하세요.
- **점수:** 판사 판정은 실행마다 조금씩 달라집니다. 자가채점은 판정 값이 아니라 표의 모양·개수만 봅니다.
- **다시 실행할 때:** 판사는 5절에서만 부릅니다(54번). 네트워크 오류로 멈추면 그 셀만 다시 실행합니다.

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"  # 사용 기록을 외부로 보내지 않음(deepeval을 불러오기 전에 설정)
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "30"  # 응답 없이 멈춘 판사 호출을 30초 뒤 다시 보냄(기본 약 88초)
import deepeval

pd.set_option("display.max_colwidth", None)  # 긴 이유 문장을 자르지 않습니다.

material_dir = Path("..")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# API 키 읽기: 단원 폴더 .env를 먼저 읽고, 없는 키만 상위 실습자료 .env에서 채웁니다
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

In [ ]:
from deepeval.models import GPTModel

# 판사 모델: temperature=0은 변동을 줄일 뿐 같은 판정을 보장하지 않습니다.
judge_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-4.1")
judge_llm = GPTModel(model=judge_model_name, temperature=0)

# 이유를 한국어로 받는 한 줄입니다. 채점 규칙이 아니므로 판단 단계 맨 끝에 붙입니다.
KO_REASON = "판정 이유(reason)는 한국어로 쓴다."

print("판사 모델:", judge_model_name, "| DeepEval 버전:", deepeval.__version__)

In [ ]:
# 퀴즈 문항·답안·교사 채점 규칙을 읽습니다.
quiz = json.loads((data_dir / "과제" / "python_quiz_set.json").read_text(encoding="utf-8"))
items = quiz["items"]
answers = [(item, answer) for item in items for answer in item["answers"]]  # (문항, 답안) 36쌍
answer_text = {answer["answer_id"]: answer["answer"] for item, answer in answers}  # 답안 ID → 답안 원문

display(pd.DataFrame([{"문항 ID": item["item_id"], "문항 유형": item["item_type"], "주제": item["topic"],
                       "기준 답변 앞부분": item["reference"][:40]} for item in items]))
print("답안:", len(answers), "| 교사 판정:", pd.Series([answer["teacher"] for item, answer in answers]).value_counts().to_dict())

In [ ]:
# 교사 채점 규칙: 문항 유형마다 기준 문장(criterion)과 판단 단계(steps). 3·4절 G-Eval의 판단 단계가 됩니다.
for kind, rule in quiz["grading_rules"].items():
    print(f"[{kind}] {rule['criterion']}")
    for number, step in enumerate(rule["steps"], start=1):
        print(f"  {number}. {step}")

In [ ]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig
from sklearn.metrics import cohen_kappa_score

run_config = {
    "async_config": AsyncConfig(max_concurrent=32),  # 동시에 보낼 케이스 수 상한. 429 오류면 8로
    "display_config": DisplayConfig(print_results=False, show_indicator=False),  # 긴 기본 출력을 끕니다.
}
BOARD_COLUMNS = ["답안 ID", "문항 ID", "문항 유형", "답안 유형", "방법", "교사", "판사", "점수", "이유"]
METHODS = ["코드 규칙", "단답 G-Eval", "서술 G-Eval(기준 답변 없음)", "서술 G-Eval"]  # 비교할 방법 4개


# [보조 함수] evaluate() 결과 → 표
def result_rows(result, info_by_id):
    """evaluate() 결과를 케이스·지표마다 한 행인 표로 바꿉니다. name(답안 ID)으로 교사 판정을 붙입니다."""
    order = {answer_id: index for index, answer_id in enumerate(info_by_id)}  # 데이터 순서로 정렬하려고 씁니다.
    rows = []
    for test_result in result.test_results:  # 케이스 하나(name = 답안 ID)
        info = info_by_id[test_result.name]
        for metric in test_result.metrics_data:  # 그 케이스에 적용한 지표 하나
            rows.append({
                "답안 ID": test_result.name, "문항 ID": info["문항 ID"], "문항 유형": info["문항 유형"], "답안 유형": info["답안 유형"],
                "방법": metric.name.replace(" [GEval]", ""),  # G-Eval 이름 뒤에 붙는 표시를 뗍니다.
                "교사": info["교사"],
                "판사": "pass" if metric.success else "fail",  # success = 점수가 threshold 이상
                "점수": round(metric.score, 2),
                "이유": metric.reason,
            })
    rows.sort(key=lambda row: (order[row["답안 ID"]], row["방법"]))
    return pd.DataFrame(rows, columns=BOARD_COLUMNS)


def scorecard(table):
    """판사 성적표: "교사"·"판사" 열(pass·fail)로 일치율·실패 탐지율·정상 통과율·kappa를 계산합니다."""
    teacher, judge = table["교사"], table["판사"]
    return {
        "일치율": round((teacher == judge).mean(), 2),
        "실패 탐지율": round((judge[teacher == "fail"] == "fail").mean(), 2),  # 교사 fail 중 판사도 fail인 비율
        "정상 통과율": round((judge[teacher == "pass"] == "pass").mean(), 2),  # 교사 pass 중 판사도 pass인 비율
        "kappa": round(cohen_kappa_score(teacher, judge), 2),
    }

## 1. 답안을 테스트 케이스로 담으세요

DeepEval 지표는 `LLMTestCase`의 필드만 읽으므로, 판사에게 보일 내용을 답안마다 필드에 나눠 담습니다.

**작성:** 함수 `make_case(item, answer)`

**목적:** 교안 02 2-3의 `make_case`를 퀴즈 답안에 적용합니다. 서술형의 채점 항목 A·B는 `context`에 담습니다.

**구현할 처리**
1. 서술형이면 `context`를 채점 항목 두 줄 목록으로, 단답이면 `None`으로 둡니다.
2. 아래 다섯 필드로 `LLMTestCase`를 만들어 반환합니다.

| 필드 | 값 |
|---|---|
| `input` | 문항 `item["question"]` |
| `actual_output` | 학생 답안 `answer["answer"]` |
| `expected_output` | 기준 답변 `item["reference"]` |
| `context` | 서술: `["채점 항목 A: " + A 문장, "채점 항목 B: " + B 문장]`(`item["rubric"]`) / 단답: `None` |
| `name` | 답안 ID `answer["answer_id"]`. 결과를 교사 판정과 잇는 열쇠 |

**반환값:** `LLMTestCase` 하나

In [ ]:
# [작성] 문항과 답안 하나로 테스트 케이스를 만드는 함수를 완성하세요.
from deepeval.test_case import LLMTestCase


def make_case(item, answer):
    """문항과 답안 하나로 LLMTestCase를 만듭니다(name은 답안 ID)."""
    # 1. 서술형이면 채점 항목 A·B 두 줄, 단답이면 None
    if item["item_type"] == "서술":
        context = [f"채점 항목 A: {item['rubric']['A']}", f"채점 항목 B: {item['rubric']['B']}"]
    else:
        context = None

    # 2. 다섯 필드로 테스트 케이스
    return LLMTestCase(
        input=item["question"],
        actual_output=answer["answer"],
        expected_output=item["reference"],
        context=context,
        name=answer["answer_id"],  # 결과를 교사 판정과 잇는 열쇠
    )

In [ ]:
# [자가채점] 답안 36개로 필드를 확인합니다(판사 호출 없음).
for item, answer in answers:
    answer_id = answer["answer_id"]
    try:
        case = make_case(item, answer)
    except Exception as error:
        raise AssertionError(f"{answer_id}: 테스트 케이스를 만들지 못했습니다. `...`를 채웠는지 확인하세요. ({type(error).__name__}: {error})")
    assert isinstance(case, LLMTestCase), "LLMTestCase를 return 하세요."
    assert case.input == item["question"], f"{answer_id}: input은 문항(question)입니다."
    assert case.actual_output == answer["answer"], f"{answer_id}: actual_output은 학생 답안(answer)입니다."
    assert case.expected_output == item["reference"], f"{answer_id}: expected_output은 기준 답변(reference)입니다."
    assert case.name == answer_id, f"{answer_id}: name은 답안 ID입니다."
    if item["item_type"] == "서술":
        expected_context = [f"채점 항목 A: {item['rubric']['A']}", f"채점 항목 B: {item['rubric']['B']}"]
        assert case.context == expected_context, f"{answer_id}: 서술형 context는 ['채점 항목 A: …', '채점 항목 B: …'] 두 줄 목록입니다."
    else:
        assert case.context is None, f"{answer_id}: 단답형 context는 None입니다."
print("테스트 케이스 확인 완료:", len(answers), "개")

In [ ]:
# 문항 유형별 케이스 목록과, 결과 표에 붙일 답안 정보(답안 ID → 문항 ID·문항 유형·답안 유형·교사)
short_cases = [make_case(item, answer) for item, answer in answers if item["item_type"] == "단답"]
essay_cases = [make_case(item, answer) for item, answer in answers if item["item_type"] == "서술"]
answer_info = {answer["answer_id"]: {"문항 ID": item["item_id"], "문항 유형": item["item_type"],
                                     "답안 유형": answer["answer_type"], "교사": answer["teacher"]}
               for item, answer in answers}
print("단답 케이스:", len(short_cases), "| 서술 케이스:", len(essay_cases))
print("서술 케이스 context 예:", essay_cases[0].context)

## 2. 단답을 코드 규칙으로 채점하세요

판사를 부르지 않는 가장 싼 방법을 기준선으로 먼저 잽니다. 교안 01 도입 표는 "정답이 하나면 문자열 비교·정규식이 더 정확"하다고 했습니다.

**작성:** 함수 `rule_grade(answer, reference)`

**목적:** 교안 01 도입 표의 단답식 채점 주의점을 퀴즈 답안으로 확인합니다. 결과는 3절 단답 G-Eval과 같은 성적표로 비교합니다.

**구현할 처리**
1. 답안과 기준 답변에서 모든 공백(띄어쓰기·줄바꿈)을 지웁니다. `"".join(text.split())`
2. 둘이 같으면 `"pass"`, 다르면 `"fail"`을 반환합니다. 대소문자는 바꾸지 않습니다(기준 답변 표기 하나만 정답).

**반환값:** `"pass"` 또는 `"fail"`

In [ ]:
# [작성] 공백을 지운 답안이 기준 답변과 같으면 pass를 반환하세요.
def rule_grade(answer, reference):
    """기준 답변 표기 하나만 정답으로 보는 코드 규칙 채점입니다."""
    # 1. 모든 공백 지우기
    answer_clean = "".join(answer.split())
    reference_clean = "".join(reference.split())

    # 2. 같으면 pass, 다르면 fail
    if answer_clean == reference_clean:
        return "pass"
    return "fail"

In [ ]:
# [자가채점] 가짜 답안으로 규칙의 경계를 확인합니다. 실제 답안은 쓰지 않습니다.
# (답안, 기준 답변, 기대 판정, 틀렸을 때 볼 곳)
fake_cases = [
    ("[1,2,3,4]", "[1, 2, 3, 4]", "pass", "띄어쓰기만 다르면 pass입니다. 양쪽 공백을 모두 지우세요."),
    (" return\n", "return", "pass", "앞뒤 공백과 줄바꿈도 지웁니다."),
    ("return", "return", "pass", "같은 표기는 pass입니다."),
    ("Return", "return", "fail", "대소문자는 바꾸지 않고 비교합니다."),
    ("return 뒤에 값을 적어요", "return", "fail", "설명을 덧붙이면 문자열이 달라 fail입니다."),
    ("리턴", "return", "fail", "한글 소리 표기도 문자열이 달라 fail입니다."),
    ("(1, 2, 3, 4)", "[1, 2, 3, 4]", "fail", "괄호가 다르면 fail입니다."),
]
for answer, reference, expected, hint in fake_cases:
    verdict = rule_grade(answer, reference)
    assert verdict in ("pass", "fail"), "\"pass\" 또는 \"fail\"을 return 하세요. `...`를 채웠는지 확인하세요."
    assert verdict == expected, f"답안 {answer!r}: {verdict!r} 대신 {expected!r}이어야 합니다. {hint}"
print("코드 규칙 확인 완료: 가짜 답안", len(fake_cases), "개 통과")

In [ ]:
# 단답 18개를 코드 규칙으로 채점해 5절 결과 표와 같은 열로 만듭니다(판사 호출 없음).
rule_rows = []
for item, answer in answers:
    if item["item_type"] != "단답":
        continue
    info = answer_info[answer["answer_id"]]
    rule_rows.append({
        "답안 ID": answer["answer_id"], "문항 ID": info["문항 ID"], "문항 유형": info["문항 유형"], "답안 유형": info["답안 유형"],
        "방법": "코드 규칙", "교사": info["교사"], "판사": rule_grade(answer["answer"], item["reference"]),
        "점수": None, "이유": "공백 제거 후 기준 답변과 비교",
    })
rule_table = pd.DataFrame(rule_rows, columns=BOARD_COLUMNS)

rule_wrong = rule_table[rule_table["판사"] != rule_table["교사"]]
print(f"교사와 다른 칸: {len(rule_wrong)} / {len(rule_table)}")
display(rule_wrong.assign(답안=rule_wrong["답안 ID"].map(answer_text))[["답안 ID", "답안", "교사", "판사"]])

- 다른 칸이 모두 교사 pass·코드 규칙 fail(헛경보)이라면, 오답은 다 잡았지만 표기가 다른 정답과 설명을 덧붙인 정답을 떨어뜨린 것입니다.
- "정답이 하나면 문자열 비교가 더 정확"은 정답 **표기**가 하나일 때 맞습니다. `리턴`, `KeyError: 'b'`처럼 교사 규칙 2번이 같은 답으로 보는 표기가 있으면 그 규칙까지 코드로 옮겨야 합니다.

## 3. 단답 G-Eval을 만드세요

교사 채점 규칙의 판단 단계를 판사에게 그대로 주면, 표기가 다른 정답과 덧붙인 설명도 규칙대로 볼 수 있습니다.

**작성:** 지표 `short_metric`

**목적:** 교안 02 3-1·3-2의 G-Eval(판단 단계 + 기준 답변)을 단답 채점에 적용합니다.

**구현할 처리**
1. `evaluation_steps`: 단답 steps 4줄(`quiz["grading_rules"]["단답"]["steps"]`) + `SHORT_REF_STEP` + `KO_REASON` 순서
2. `evaluation_params`: 문항·답안·기준 답변(`INPUT`, `ACTUAL_OUTPUT`, `EXPECTED_OUTPUT`). 단답 케이스는 `context`가 없으므로 `CONTEXT`는 넣지 않습니다.
3. `name`, `model`, `threshold=0.7`은 골격에 있습니다.

**반환값:** `GEval` 지표 하나(이 셀에서는 판사를 부르지 않습니다)

In [ ]:
# [작성] 단답 채점 규칙으로 G-Eval 지표를 만드세요.
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

SHORT_REF_STEP = "expected_output은 모범 답안이다. 학생 답안의 답이 그와 같은 답인지 본다."

short_metric = GEval(
    name="단답 G-Eval",
    evaluation_steps=quiz["grading_rules"]["단답"]["steps"] + [SHORT_REF_STEP, KO_REASON],  # 1. KO_REASON은 맨 끝
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                       SingleTurnParams.EXPECTED_OUTPUT],  # 2. 단답 케이스에는 context가 없습니다.
    model=judge_llm,
    threshold=0.7,
)

In [ ]:
# [자가채점] 지표 설정을 확인합니다(판사 호출 없음).
INPUT, ACTUAL_OUTPUT = SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT
EXPECTED_OUTPUT, CONTEXT = SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.CONTEXT

short_steps = quiz["grading_rules"]["단답"]["steps"]
assert short_metric.name == "단답 G-Eval", "name은 \"단답 G-Eval\"입니다. 5절 결과 표의 방법 이름이 됩니다."
assert short_metric.evaluation_steps == short_steps + [SHORT_REF_STEP, KO_REASON], "판단 단계는 단답 steps 4줄, SHORT_REF_STEP, KO_REASON 순서입니다(KO_REASON은 맨 끝)."
assert isinstance(short_metric.evaluation_params, list), "evaluation_params는 SingleTurnParams 목록입니다. `...`를 채웠는지 확인하세요."
assert set(short_metric.evaluation_params) == {INPUT, ACTUAL_OUTPUT, EXPECTED_OUTPUT}, "판사에게 보일 필드는 INPUT·ACTUAL_OUTPUT·EXPECTED_OUTPUT입니다. 단답 케이스에는 CONTEXT가 없습니다."
assert short_metric.threshold == 0.7, "threshold는 0.7입니다."
print("단답 G-Eval 확인 완료")

## 4. 서술형 rubric G-Eval 두 개를 만드세요

서술형은 채점 항목 A·B를 몇 개 충족했는지가 점수이므로 rubric으로 점수 구간을 정하고, 기준 답변을 줄 때와 안 줄 때를 비교합니다.

| `score_range` | `expected_outcome` |
|---|---|
| (7, 10) | A·B를 모두 충족했다 |
| (4, 6) | A·B 가운데 하나만 충족했다 |
| (0, 3) | 채점 항목 A·B를 모두 충족하지 못했다 |

- threshold 0.65: 7점(0.7)부터 통과, 6점(0.6)은 불통과입니다. 7점이 0.68처럼 나와도 통과합니다(교안 02 3-3).
- `ESSAY_STEPS`: 서술 steps 4줄 앞에 "context의 채점 항목 A·B를 기준으로 본다." 한 줄을 붙인 판단 단계입니다.

**작성:** 지표 `essay_metric_noref`, `essay_metric_ref`

**목적:** 교안 02 3-2(기준 답변 있이·없이)와 3-3(rubric)을 함께 적용합니다. 한 번에 한 가지만 바꿔야 차이의 원인을 압니다. 두 지표는 기준 답변을 주는지만 다릅니다.

**구현할 처리**
1. `essay_metric_noref`: steps `ESSAY_STEPS + [KO_REASON]`, params `INPUT`·`ACTUAL_OUTPUT`·`CONTEXT`
2. `essay_metric_ref`: 1과 같고, steps의 `KO_REASON` 바로 앞에 `ESSAY_REF_STEP`, params에 `EXPECTED_OUTPUT`만 더합니다.
3. `name`, `rubric`, `model`, `threshold`는 골격에 있습니다.

**반환값:** `GEval` 지표 두 개

In [ ]:
from deepeval.metrics.g_eval import Rubric

# 충족한 채점 항목 수를 0–10 구간으로 옮깁니다(구간은 겹치면 안 됩니다).
ESSAY_RUBRIC = [
    Rubric(score_range=(0, 3), expected_outcome="채점 항목 A·B를 모두 충족하지 못했다"),
    Rubric(score_range=(4, 6), expected_outcome="A·B 가운데 하나만 충족했다"),
    Rubric(score_range=(7, 10), expected_outcome="A·B를 모두 충족했다"),
]
ESSAY_STEPS = ["context의 채점 항목 A·B를 기준으로 본다."] + quiz["grading_rules"]["서술"]["steps"]
ESSAY_REF_STEP = "expected_output은 모범 답안의 예다. 표현이 달라도 같은 내용이면 충족이다."

In [ ]:
# [작성] 기준 답변 없이·있이 서술 G-Eval 두 개를 만드세요.
# 1. 기준 답변 없이: 문항·답안·채점 항목
essay_metric_noref = GEval(
    name="서술 G-Eval(기준 답변 없음)",
    evaluation_steps=ESSAY_STEPS + [KO_REASON],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    rubric=ESSAY_RUBRIC,
    model=judge_llm,
    threshold=0.65,
)

# 2. 기준 답변 있이: 1과 같고 ESSAY_REF_STEP 한 줄과 EXPECTED_OUTPUT만 더함
essay_metric_ref = GEval(
    name="서술 G-Eval",
    evaluation_steps=ESSAY_STEPS + [ESSAY_REF_STEP, KO_REASON],  # KO_REASON은 맨 끝
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT,
                       SingleTurnParams.EXPECTED_OUTPUT],
    rubric=ESSAY_RUBRIC,
    model=judge_llm,
    threshold=0.65,
)

In [ ]:
# [자가채점] 두 지표의 차이가 EXPECTED_OUTPUT과 판단 단계 한 줄뿐인지 확인합니다(판사 호출 없음).
base_params = {INPUT, ACTUAL_OUTPUT, CONTEXT}
assert essay_metric_noref.name == "서술 G-Eval(기준 답변 없음)" and essay_metric_ref.name == "서술 G-Eval", "name은 골격 그대로 둡니다."
assert essay_metric_noref.evaluation_steps == ESSAY_STEPS + [KO_REASON], "기준 답변 없는 지표의 판단 단계는 ESSAY_STEPS + [KO_REASON]입니다."
assert essay_metric_ref.evaluation_steps == ESSAY_STEPS + [ESSAY_REF_STEP, KO_REASON], "기준 답변 있는 지표는 KO_REASON 바로 앞에 ESSAY_REF_STEP 한 줄만 더합니다."
for metric in (essay_metric_noref, essay_metric_ref):
    assert isinstance(metric.evaluation_params, list), f"{metric.name}: evaluation_params는 SingleTurnParams 목록입니다. `...`를 채웠는지 확인하세요."
assert set(essay_metric_noref.evaluation_params) == base_params, "기준 답변 없는 지표는 INPUT·ACTUAL_OUTPUT·CONTEXT만 봅니다."
assert set(essay_metric_ref.evaluation_params) == base_params | {EXPECTED_OUTPUT}, "기준 답변 있는 지표는 여기에 EXPECTED_OUTPUT 하나만 더합니다."
for metric in (essay_metric_noref, essay_metric_ref):
    assert metric.rubric == ESSAY_RUBRIC and metric.threshold == 0.65, f"{metric.name}: rubric과 threshold는 골격 그대로 둡니다."
print("서술 G-Eval 두 개 확인 완료: 차이는 EXPECTED_OUTPUT과 판단 단계 한 줄")

## 5. evaluate로 채점하고 결과 표를 합치세요

방법 네 개를 같은 성적표로 비교하려면 결과를 한 표(방법 4개 × 답안 18개 = 72행)에 모아야 합니다.

**작성:** `short_result`, `essay_result`, 결과 표 `board`

**목적:** 교안 02 3-2의 `evaluate` → `result_rows` 흐름을 퀴즈 답안에 적용합니다.

**구현할 처리**
1. 단답 케이스 18개를 `short_metric`으로 `evaluate`합니다(판사 18번). `**run_config`를 함께 넘깁니다.
2. 서술 케이스 18개를 서술 지표 두 개로 한 번에 `evaluate`합니다(판사 36번).
3. 두 결과를 `result_rows(결과, answer_info)`로 표로 바꿉니다.
4. `rule_table`, 단답 표, 서술 표를 이 순서로 위아래로 붙입니다(`pd.concat(…, ignore_index=True)`).

**반환값:** `board`(72행, 열은 `BOARD_COLUMNS`)

In [ ]:
# [작성] 단답·서술 케이스를 evaluate로 채점하고 코드 규칙 표와 합치세요. 판사 호출 54번
# 1. 단답 18개 × 단답 G-Eval (판사 18번)
short_result = evaluate(test_cases=short_cases, metrics=[short_metric], **run_config)

# 2. 서술 18개 × 서술 G-Eval 두 개 (판사 36번)
essay_result = evaluate(test_cases=essay_cases, metrics=[essay_metric_noref, essay_metric_ref], **run_config)

# 3. 결과 → 표
short_table = result_rows(short_result, answer_info)
essay_table = result_rows(essay_result, answer_info)

# 4. 코드 규칙·단답·서술 표를 위아래로 붙이기 (72행)
board = pd.concat([rule_table, short_table, essay_table], ignore_index=True)

In [ ]:
# [자가채점] 방법별 행 수·열·교사 판정을 확인합니다. 판사 판정 값은 실행마다 달라지므로 보지 않습니다.
METHOD_KIND = {"코드 규칙": "단답", "단답 G-Eval": "단답", "서술 G-Eval(기준 답변 없음)": "서술", "서술 G-Eval": "서술"}
assert isinstance(board, pd.DataFrame), "board는 DataFrame입니다. `...`를 채웠는지 확인하세요."
assert list(board.columns) == BOARD_COLUMNS, f"board의 열은 {BOARD_COLUMNS}입니다. 표는 rule_table과 result_rows 결과만 붙입니다."
assert len(board) == 72, f"board는 72행(방법 4개 × 18)이어야 합니다. 지금 {len(board)}행입니다."
assert list(board["방법"].iloc[[0, 18, 36]]) == ["코드 규칙", "단답 G-Eval", "서술 G-Eval"], "코드 규칙, 단답, 서술 표 순서로 붙이세요."
for method, kind in METHOD_KIND.items():
    rows = board[board["방법"] == method]
    assert len(rows) == 18, f"{method}: 18행이어야 합니다. 지표 name과 evaluate에 넘긴 지표·케이스를 확인하세요."
    assert set(rows["문항 유형"]) == {kind} and rows["답안 ID"].is_unique, f"{method}: {kind} 답안 18개를 한 번씩 채점해야 합니다."
assert board["판사"].isin(["pass", "fail"]).all(), "판사 열은 pass·fail입니다."
assert (board["교사"] == board["답안 ID"].map(lambda answer_id: answer_info[answer_id]["교사"])).all(), "교사 열은 answer_info의 교사 판정입니다."
judge_scores = board.loc[board["방법"] != "코드 규칙", "점수"]
assert judge_scores.between(0, 1).all(), "G-Eval 점수는 0–1입니다."
print("결과 표 확인 완료:", board.shape, "| 방법별 행 수:", board["방법"].value_counts().reindex(METHODS).to_dict())

In [ ]:
# 방법마다 교사와 다른 칸을 이유와 함께 봅니다.
wrong = board[board["판사"] != board["교사"]]
print("교사와 다른 칸 수:", wrong["방법"].value_counts().reindex(METHODS, fill_value=0).to_dict())
display(wrong[["답안 ID", "방법", "교사", "판사", "점수", "이유"]])

In [ ]:
# 서술 두 지표를 답안마다 나란히 봅니다. 판정이 갈린 답안이 기준 답변을 준 효과입니다.
essay_board = board[board["문항 유형"] == "서술"]
display(essay_board.pivot(index=["답안 ID", "교사"], columns="방법", values=["점수", "판사"]))

- 서술 두 지표의 판정이 갈린 답안은 `이유`에서 판사가 기준 답변을 어떻게 썼는지 읽습니다.
- `다른_표현_정답`이 기준 답변을 준 쪽에서만 fail이라면, 기준 답변과 표현이 다르다고 감점한 것입니다.
- `길지만_틀림`이 pass라면 긴 답안을 후하게 본 것입니다(교안 01 5절 장황함 효과).

## 6. 방법별 성적표와 답안 유형 표를 만드세요

일치율 하나로는 놓친 실패와 헛경보를 가를 수 없으므로, 방법마다 성적표 4지표를 보고 어느 답안 유형에서 틀렸는지 표로 좁힙니다.

**작성:** 방법별 성적표 `method_cards`, 답안 유형 표 `type_table`

**목적:** 교안 02 6-2의 판사 성적표와 6-3의 결함×방법 표를 방법 4개에 적용합니다.

**구현할 처리**
1. 방법마다 `board`에서 그 방법 행만 골라 `scorecard`를 계산해 `cards[방법]`에 넣고, `pd.DataFrame(cards).T`로 행이 방법인 표를 만듭니다.
2. `flags`(골격에 있음)를 (`문항 유형`, `답안 유형`, `방법`)별로 묶어 `판사 fail`·`교사 fail` 개수를 더합니다.
3. 칸을 `"판사 fail 수 / 교사 fail 수"` 문자열로 만들고 `unstack()`으로 방법을 열로 펼친 뒤, `reindex(index=TYPE_ORDER, columns=METHODS)`로 순서를 맞추고 빈 칸은 `"–"`로 채웁니다.

**반환값:** `method_cards`(4행 × 4열), `type_table`(12행 × 4열)

In [ ]:
# [작성] 방법별 성적표와 답안 유형 표를 만드세요.
# 1. 방법마다 성적표
cards = {}
for method in METHODS:
    cards[method] = scorecard(board[board["방법"] == method])
method_cards = pd.DataFrame(cards).T  # cards → 행이 방법인 표

# 2. (문항 유형, 답안 유형, 방법)별 판사 fail 수·교사 fail 수
flags = board[["문항 유형", "답안 유형", "방법"]].copy()
flags["판사 fail"] = board["판사"] == "fail"
flags["교사 fail"] = board["교사"] == "fail"
counts = flags.groupby(["문항 유형", "답안 유형", "방법"])[["판사 fail", "교사 fail"]].sum()

# 3. "판사 fail 수 / 교사 fail 수" 칸 (행: 문항 유형·답안 유형, 열: 방법)
TYPE_ORDER = [(kind, answer_type) for kind, types in quiz["answer_types"].items() for answer_type in types]
pairing = counts["판사 fail"].astype(str) + " / " + counts["교사 fail"].astype(str)
type_table = pairing.unstack().reindex(index=TYPE_ORDER, columns=METHODS).fillna("–")

In [ ]:
# [자가채점] 성적표와 답안 유형 표를 board에서 다시 세어 대조합니다.
CARD_COLUMNS = ["일치율", "실패 탐지율", "정상 통과율", "kappa"]
assert isinstance(method_cards, pd.DataFrame) and list(method_cards.index) == METHODS, "method_cards의 행은 METHODS 순서의 방법 4개입니다. `...`를 채웠는지 확인하세요."
assert list(method_cards.columns) == CARD_COLUMNS, f"method_cards의 열은 {CARD_COLUMNS}입니다(scorecard 결과)."
for method in METHODS:
    rows = board[board["방법"] == method]
    teacher_fail = rows[rows["교사"] == "fail"]
    teacher_pass = rows[rows["교사"] == "pass"]
    expected = {
        "일치율": round((rows["교사"] == rows["판사"]).sum() / len(rows), 2),
        "실패 탐지율": round((teacher_fail["판사"] == "fail").sum() / len(teacher_fail), 2),
        "정상 통과율": round((teacher_pass["판사"] == "pass").sum() / len(teacher_pass), 2),
    }
    for column, value in expected.items():
        assert abs(method_cards.loc[method, column] - value) < 1e-9, f"{method} {column}: 그 방법의 행 18개만 골라 scorecard를 계산하세요."

assert isinstance(type_table, pd.DataFrame) and type_table.shape == (12, 4), "type_table은 12행(문항 유형·답안 유형) × 4열(방법)입니다."
assert list(type_table.index) == TYPE_ORDER and list(type_table.columns) == METHODS, "reindex로 행은 TYPE_ORDER, 열은 METHODS 순서로 맞추세요."
for (kind, answer_type), method in [(key, method) for key in TYPE_ORDER for method in METHODS]:
    rows = board[(board["문항 유형"] == kind) & (board["답안 유형"] == answer_type) & (board["방법"] == method)]
    expected = "–" if len(rows) == 0 else f"{(rows['판사'] == 'fail').sum()} / {(rows['교사'] == 'fail').sum()}"
    assert type_table.loc[(kind, answer_type), method] == expected, f"{kind} {answer_type} × {method}: {expected!r}이어야 합니다(그 방법을 쓰지 않은 칸은 '–')."
print("성적표·답안 유형 표 확인 완료")

In [ ]:
display(method_cards)
print("칸 = 판사 fail 수 / 교사 fail 수 (–는 그 방법을 쓰지 않은 칸)")
display(type_table)

- 교사 fail은 단답 `오개념`·`근접_오답`·`여러_답`, 서술 `길지만_틀림`·`A만_충족`·`오개념` 행에만 있습니다. 다른 행의 판사 fail은 헛경보입니다.
- 교사 fail 행에서 판사 fail 수가 3보다 작으면 놓친 실패입니다. 서술 `A만_충족`을 놓쳤다면 B가 빠진 것을 보지 못한 것입니다.

## 7. 평가 범위표로 다음 주에 보탤 답안 유형을 고르세요

성적표는 답안 36개가 다룬 유형 안에서만 믿을 수 있으므로, 목표 유형(`coverage_targets`) 가운데 빠진 유형을 찾습니다.

**작성:** 평가 범위표 `coverage_table`

**목적:** 교안 02 7-1의 평가 범위표(유형별 수와 `보강 후보`)를 답안 유형에 적용합니다.

**구현할 처리**
1. 문항 유형마다 `coverage_targets`의 답안 유형을 순서대로 읽습니다(골격에 있음).
2. `answer_info`에서 그 문항 유형·답안 유형인 답안 수를 셉니다.
3. 답안 수가 0이면 `"보강 후보"`, 아니면 `"있음"`을 상태로 씁니다.

**반환값:** `coverage_table`(16행, 열 `문항 유형`, `답안 유형`, `답안 수`, `상태`)

In [ ]:
# [작성] 목표 답안 유형마다 답안 수와 상태를 세어 평가 범위표를 만드세요.
coverage_rows = []
for kind, target_types in quiz["coverage_targets"].items():
    for answer_type in target_types:
        # 2. 그 문항 유형·답안 유형인 답안 수
        count = 0
        for info in answer_info.values():
            if info["문항 유형"] == kind and info["답안 유형"] == answer_type:
                count += 1

        # 3. 0개면 보강 후보
        status = "보강 후보" if count == 0 else "있음"

        coverage_rows.append({"문항 유형": kind, "답안 유형": answer_type, "답안 수": count, "상태": status})
coverage_table = pd.DataFrame(coverage_rows)

In [ ]:
# [자가채점] 답안 목록에서 다시 세어 대조합니다.
coverage_columns = ["문항 유형", "답안 유형", "답안 수", "상태"]
assert isinstance(coverage_table, pd.DataFrame) and list(coverage_table.columns) == coverage_columns, f"coverage_table의 열은 {coverage_columns}입니다."
targets = [(kind, answer_type) for kind, types in quiz["coverage_targets"].items() for answer_type in types]
assert list(zip(coverage_table["문항 유형"], coverage_table["답안 유형"])) == targets, "행은 coverage_targets의 순서대로 16개입니다."
for row in coverage_table.itertuples(index=False):
    expected = sum(1 for item, answer in answers if item["item_type"] == row[0] and answer["answer_type"] == row[1])
    assert row[2] == expected, f"{row[0]} {row[1]}: 답안 수는 {expected}개입니다. `...`를 채웠는지 확인하세요."
    assert row[3] == ("보강 후보" if expected == 0 else "있음"), f"{row[0]} {row[1]}: 0개면 \"보강 후보\", 아니면 \"있음\"입니다."
print("평가 범위표 확인 완료: 보강 후보", int((coverage_table["상태"] == "보강 후보").sum()), "개")

In [ ]:
display(coverage_table)

보강 후보는 답안부터 만들 수 없습니다. 교사 채점 규칙이 그 답안을 어떻게 판정할지 먼저 정해야 교사 판정을 붙일 수 있습니다.

| 보강 후보 | 지금 규칙으로는 | 먼저 할 일 |
|---|---|---|
| 단답 `대소문자_다름`(예: `keyerror`) | 규칙에 없음 | 대소문자 차이를 같은 답으로 볼지 정하기(파이썬 이름은 대소문자를 구분) |
| 단답 `틀린_설명_덧붙임`(예: `return. 값을 화면에 출력해요`) | 규칙 1 "덧붙인 설명은 감점하지 않는다"로 pass | 틀린 설명을 감점할지 정하고 규칙 1 고치기 |
| 서술 `B만_충족`, `질문과_다른_답` | A·B를 다 충족하지 못해 fail | 규칙은 분명함. 문항마다 답안과 교사 판정 만들기 |

규칙과 답안을 더한 뒤 5–6절을 다시 실행해 성적표를 다시 잽니다.

## 8. 문항 유형별 채점 방법을 고르세요

성적표를 볼 때마다 기준이 바뀌지 않게, 운영팀이 정한 규칙을 함수로 고정해 다음 주 퀴즈에도 같은 방식으로 고릅니다.

**규칙** (운영팀이 정한 값이며 교안에 정해진 값이 아닙니다)
- 실패 탐지율 ≥ 0.8 **그리고** 정상 통과율 ≥ 0.9인 방법만 채택 후보입니다. 교사 fail·pass가 9개씩이므로 놓친 실패 1건 이하, 헛경보 0건입니다.
- 헛경보를 더 엄격하게 막습니다. 맞은 답을 틀렸다고 하면 학생이 손해를 보고 이의 신청이 늘어납니다.
- 후보가 여럿이면 목록 앞쪽(싸고 단순한 방법)을 고릅니다. 단답: 코드 규칙 → 단답 G-Eval, 서술: 서술 G-Eval(기준 답변 없음) → 서술 G-Eval
- 후보가 없으면 `"교사 채점"`입니다. 일치율·kappa는 판정에 쓰지 않습니다.

**작성:** 함수 `choose_method(cards, candidates)`

**목적:** 교안 02 6-2의 성적표를 규칙으로 읽어, 교안 02 8-3처럼 채택 기준을 넘는 지표만 씁니다.

**구현할 처리**
1. `candidates`를 앞에서부터 보며 `cards.loc[방법, "실패 탐지율"]`, `cards.loc[방법, "정상 통과율"]`을 읽습니다.
2. 두 값이 모두 기준 이상인 첫 방법 이름을 반환합니다. 기준값과 같으면 통과입니다.
3. 끝까지 없으면 `"교사 채점"`을 반환합니다.

**반환값:** 방법 이름 또는 `"교사 채점"`

In [ ]:
# [작성] 성적표와 후보 목록으로 채점 방법을 고르는 함수를 완성하세요.
FAIL_CATCH_MIN = 0.8  # 실패 탐지율 하한: 교사 fail 9개 중 놓친 실패 1건 이하
PASS_KEEP_MIN = 0.9   # 정상 통과율 하한: 교사 pass 9개 중 헛경보 0건


def choose_method(cards, candidates):
    """후보를 앞에서부터 보고 두 기준을 넘는 첫 방법 이름을 반환합니다. 없으면 "교사 채점"."""
    for method in candidates:
        # 1. 그 방법의 실패 탐지율과 정상 통과율
        fail_catch = cards.loc[method, "실패 탐지율"]
        pass_keep = cards.loc[method, "정상 통과율"]

        # 2. 두 기준을 모두 넘으면 그 방법
        if fail_catch >= FAIL_CATCH_MIN and pass_keep >= PASS_KEEP_MIN:
            return method

    # 3. 넘는 방법이 없으면
    return "교사 채점"

In [ ]:
# [자가채점] 가짜 성적표로 규칙의 순서와 경계를 확인합니다. 실제 성적표는 쓰지 않습니다.
def fake_cards(**scores):
    """방법=(실패 탐지율, 정상 통과율)로 가짜 성적표를 만듭니다. 일치율·kappa는 판정과 무관하게 높게 둡니다."""
    return pd.DataFrame({method: {"일치율": 0.95, "실패 탐지율": catch, "정상 통과율": keep, "kappa": 0.9}
                         for method, (catch, keep) in scores.items()}).T


# (이름, 가짜 성적표, 후보, 기대 결과, 틀렸을 때 볼 규칙)
fake_tables = [
    ("가 둘 다 통과", fake_cards(규칙=(1.0, 1.0), 판사=(1.0, 1.0)), ["규칙", "판사"], "규칙", "후보가 여럿이면 목록 앞쪽 방법입니다."),
    ("나 앞쪽 헛경보 많음", fake_cards(규칙=(1.0, 0.44), 판사=(0.89, 1.0)), ["규칙", "판사"], "판사", "정상 통과율이 0.9 미만이면 다음 후보를 봅니다."),
    ("다 경계값과 같음", fake_cards(규칙=(0.8, 0.9), 판사=(1.0, 1.0)), ["규칙", "판사"], "규칙", "기준값과 같으면 통과입니다(>=)."),
    ("라 실패 탐지율 0.78", fake_cards(규칙=(0.78, 1.0), 판사=(0.8, 0.89)), ["규칙", "판사"], "교사 채점", "실패 탐지율 0.78, 정상 통과율 0.89는 기준 미만입니다. 후보가 없으면 \"교사 채점\"입니다."),
    ("마 후보 순서", fake_cards(규칙=(1.0, 1.0), 판사=(1.0, 1.0)), ["판사", "규칙"], "판사", "성적표의 행 순서가 아니라 candidates 순서로 봅니다."),
]
for table_name, cards_table, candidates, expected, rule in fake_tables:
    choice = choose_method(cards_table, candidates)
    assert choice == expected, f"[{table_name}] {choice!r} 대신 {expected!r}이어야 합니다. {rule}"
print("방법 고르기 함수 확인 완료: 가짜 성적표", len(fake_tables), "개 통과")

In [ ]:
# 실제 성적표로 문항 유형마다 채점 방법을 고릅니다. 후보는 앞쪽이 싸고 단순한 방법입니다.
CANDIDATES = {"단답": ["코드 규칙", "단답 G-Eval"], "서술": ["서술 G-Eval(기준 답변 없음)", "서술 G-Eval"]}
for kind, candidates in CANDIDATES.items():
    print(f"{kind} → {choose_method(method_cards, candidates)}")
display(method_cards[["실패 탐지율", "정상 통과율"]])

## 9. 결정 기록을 쓰세요

**작성:** 아래 마크다운 셀을 더블클릭해 괄호 안 안내를 지우고 제목과 네 칸을 채웁니다. 숫자는 자기 실행 결과에서 옮깁니다.

**넣을 내용**
1. 상태: 제안·채택·거절·대체됨 가운데 하나
2. 맥락(2–3문장): 정할 것(문항 유형별 1차 채점 방법), 평가 방법(답안 36개, 방법 4개, 채택 규칙). 결정은 쓰지 않습니다.
3. 결정: 8절 결과를 "…한다."로 쓰고, `근거:`에 줄마다 숫자 하나 이상
    - 방법별 실패 탐지율·정상 통과율
    - 코드 규칙이 떨어뜨린 정답 유형(헛경보)
    - 기준 답변을 줄 때와 안 줄 때 판정이 갈린 답안
    - 평가 범위표의 보강 후보와 먼저 정할 교사 규칙
4. 결과: `얻는 것:`, `감수하는 것:`, `다시 검토할 조건:`(바뀌는 것 → 다시 할 일, 둘 이상)

#### 결정 기록: 단답은 단답 G-Eval, 서술은 기준 답변을 준 서술 G-Eval로 1차 채점

(아래 숫자는 정답 예시 실행 한 번의 출력입니다. 판사 판정은 실행마다 조금씩 달라집니다. 코드 규칙과 평가 범위표의 숫자는 실행마다 같습니다.)

- **상태**: 채택
- **맥락**: 부트캠프 운영팀은 매주 바뀌는 파이썬 퀴즈의 1차 채점을 문항 유형마다 어떤 방법으로 할지 정해야 한다. 조교가 채점한 답안 36개(단답 18·서술 18)를 방법 4개(코드 규칙, 단답 G-Eval, 서술 G-Eval 기준 답변 없음·있음)로 채점해 교사 판정과 대조했다. 실패 탐지율 0.8 이상, 정상 통과율 0.9 이상인 방법 가운데 목록 앞쪽(싸고 단순한 쪽)을 고른다.
- **결정**: 단답은 단답 G-Eval, 서술은 기준 답변을 준 서술 G-Eval로 1차 채점하고, 교사는 판사 fail 답안과 이의 신청만 확인한다.
    - 근거:
        - 코드 규칙은 실패 탐지율 1.0이지만 정상 통과율 0.44(교사 pass 9개 중 4개)라 탈락했다. 떨어뜨린 정답 5개는 `표기_다름` 2개(`리턴`, `KeyError: 'b'`)와 `설명_덧붙임` 3개다. 띄어쓰기만 다른 `[1,2,3,4]`는 통과했다.
        - 단답 G-Eval: 실패 탐지율 (실행 후 채움: 값), 정상 통과율 (실행 후 채움: 값). 교사와 다른 칸 (실행 후 채움: 답안 ID와 이유 요약).
        - 서술 G-Eval(기준 답변 없음): 실패 탐지율 (실행 후 채움: 값), 정상 통과율 (실행 후 채움: 값). 서술 G-Eval: 실패 탐지율 (실행 후 채움: 값), 정상 통과율 (실행 후 채움: 값).
        - 기준 답변 유무로 판정이 갈린 답안: (실행 후 채움: 답안 ID, 두 점수, 이유에서 기준 답변을 쓴 부분. 없으면 없다고 쓰고 점수 차이가 가장 큰 답안).
        - 평가 범위표: 목표 16유형 중 보강 후보 4개(단답 `대소문자_다름`·`틀린_설명_덧붙임`, 서술 `B만_충족`·`질문과_다른_답`). `틀린_설명_덧붙임`은 지금 규칙 1 "덧붙인 설명은 감점하지 않는다"와 부딪혀 규칙부터 정해야 한다.
- **결과**
    - 얻는 것: 단답 18개·서술 18개의 1차 채점을 판사가 맡고, 같은 테스트 케이스와 지표로 매주 같은 채점을 반복한다.
    - 감수하는 것: 매주 판사 호출 비용(이번 54번)이 들고, 판사 판정이 실행마다 조금씩 달라 기준값 가까운 점수는 교사가 다시 본다. 보강 후보 4유형에서는 판사 성적을 아직 모른다.
    - 다시 검토할 조건:
        - 다음 주 퀴즈가 나오면 → 교사가 일부 답안을 채점해 같은 셀로 회귀 평가하고 성적표가 채택 기준을 넘는지 확인한다.
        - 판사 모델(`OPENAI_EVAL_MODEL`)을 바꾸면 → 5–6절을 다시 실행해 성적표부터 다시 잰다.
        - 대소문자·틀린 설명 규칙을 정하면 → 보강 후보 답안과 교사 판정을 더하고 6–8절을 다시 실행한다.